In [ ]:

import pandas as pd

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        print(f"\nDataFrame: {name}")
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())


In [ ]:

%whos



In [ ]:
pd.read_csv(...)

In [ ]:

import pandas as pd

df = pd.read_csv('/content/facilities.csv')

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())


In [ ]:

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)

print("All columns:")
for col in df.columns:
    print(col)

print("\nBorough distribution:")
print(df['boro'].value_counts(dropna=False))

print("\nSample facility names:")
print(df['facname'].value_counts().head(20))


In [ ]:

print("FACILITY GROUPS:")
print(df['facgroup'].value_counts(dropna=False).head(30))

print("\nFACILITY SUBGROUPS:")
print(df['facsubgrp'].value_counts(dropna=False).head(30))

print("\nFACILITY TYPES:")
print(df['factype'].value_counts(dropna=False).head(30))

print("\nCENSUS TRACT SAMPLE:")
print(df[['boro', 'ct2020']].head(10))


In [ ]:

print("HEALTHCARE SUBCATEGORIES:")
print(
    df[df['facgroup'] == 'HEALTH CARE']['facsubgrp']
    .value_counts(dropna=False)
)

print("\nPARKS SUBCATEGORIES:")
print(
    df[df['facgroup'] == 'PARKS AND PLAZAS']['facsubgrp']
    .value_counts(dropna=False)
)

print("\nSCHOOL SUBCATEGORIES:")
print(
    df[df['facgroup'].isin([
        'SCHOOLS (K-12)',
        'HIGHER EDUCATION',
        'VOCATIONAL AND PROPRIETARY SCHOOLS'
    ])]['facsubgrp'].value_counts(dropna=False)
)


In [ ]:

categories = {
    'Healthcare': df['facgroup'] == 'HEALTH CARE',
    'Hospitals_Clinics': df['facsubgrp'] == 'HOSPITALS AND CLINICS',
    'Parks_Plazas': df['facgroup'] == 'PARKS AND PLAZAS',
    'Dedicated_Parks': df['facsubgrp'] == 'PARKS',
    'K12_Schools': df['facgroup'] == 'SCHOOLS (K-12)',
    'Higher_Education': df['facgroup'] == 'HIGHER EDUCATION'
}

borough_counts = pd.DataFrame(index=sorted(df['boro'].dropna().unique()))

for category, condition in categories.items():
    counts = df[condition].groupby('boro').size()
    borough_counts[category] = counts

borough_counts = borough_counts.fillna(0).astype(int)

display(borough_counts)


In [ ]:

population = {
    'BRONX': 1472654,
    'BROOKLYN': 2736074,
    'MANHATTAN': 1694251,
    'QUEENS': 2405464,
    'STATEN ISLAND': 495747
}

borough_counts['Population'] = pd.Series(population)

facility_columns = list(categories.keys())

per_capita = borough_counts.copy()

for col in facility_columns:
    per_capita[col] = (
        per_capita[col] / per_capita['Population']
    ) * 10000

print("FACILITIES PER 10,000 RESIDENTS")
display(per_capita.round(2))


In [ ]:

import matplotlib.pyplot as plt

plot_data = per_capita[facility_columns]

ax = plot_data.plot(
    kind='bar',
    figsize=(14, 7),
    width=0.8
)

plt.title('Public Facilities per 10,000 Residents by NYC Borough',
          fontsize=15)
plt.xlabel('NYC Borough')
plt.ylabel('Facilities per 10,000 Residents')
plt.xticks(rotation=0)
plt.legend(title='Facility Category', bbox_to_anchor=(1.02, 1))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

sns.heatmap(
    per_capita[facility_columns],
    annot=True,
    fmt='.2f',
    cmap='YlGnBu',
    linewidths=0.5
)

plt.title('NYC Public Facility Availability per 10,000 Residents',
          fontsize=15)
plt.xlabel('Facility Category')
plt.ylabel('Borough')
plt.tight_layout()
plt.show()


In [ ]:

categories = {
    'Healthcare': df['facgroup'] == 'HEALTH CARE',

    'Hospitals_Clinics': (
        (df['facgroup'] == 'HEALTH CARE') &
        (df['facsubgrp'] == 'HOSPITALS AND CLINICS')
    ),

    'Parks_Plazas': df['facgroup'] == 'PARKS AND PLAZAS',

    'Dedicated_Parks': (
        (df['facgroup'] == 'PARKS AND PLAZAS') &
        (df['facsubgrp'] == 'PARKS')
    ),

    'K12_Schools': df['facgroup'] == 'SCHOOLS (K-12)',

    'Higher_Education': df['facgroup'] == 'HIGHER EDUCATION'
}

borough_counts = pd.DataFrame(
    index=sorted(df['boro'].dropna().unique())
)

for category, condition in categories.items():
    borough_counts[category] = (
        df[condition].groupby('boro').size()
    )

borough_counts = borough_counts.fillna(0).astype(int)

borough_counts['Population'] = pd.Series(population)

facility_columns = list(categories.keys())

per_capita = borough_counts.copy()

for col in facility_columns:
    per_capita[col] = (
        per_capita[col] / per_capita['Population'] * 10000
    )

display(per_capita.round(2))


In [ ]:

borough_counts.to_csv('borough_facility_counts.csv')
per_capita.round(2).to_csv('borough_facilities_per_10000.csv')

print("Files saved successfully!")
